In [26]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
import json


In [27]:
# Generate a synthetic dataset with 50 input features
# Let's create 1000 samples with 50 features each and binary labels
np.random.seed(42)  # For reproducibility

X = np.random.rand(1000, 50)  # 1000 samples, 50 features
y = np.random.randint(2, size=(1000, 1))  # Binary labels (0 or 1)

print(X.shape, y.shape)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# print(y_train.shape, y_test.shape)
# print(X_train.shape, X_test.shape)


(1000, 50) (1000, 1)


In [28]:
# Define a single-layer perceptron model
model = Sequential([
    Dense(1, input_dim=50, activation='relu'),  # One neuron, 50 inputs, sigmoid activation
    # BatchNormalization()
])

# Compile the model
model.compile(optimizer='sgd',  # Stochastic Gradient Descent
              loss='binary_crossentropy',  # Loss function for binary classification
              metrics=['accuracy'])


# Train the model
model.fit(X_train, y_train, epochs=20, batch_size=10, verbose=1)


Epoch 1/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 390us/step - accuracy: 0.5065 - loss: 7.9545 
Epoch 2/20


/home/nestor/.local/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 291us/step - accuracy: 0.4781 - loss: 8.4127
Epoch 3/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 476us/step - accuracy: 0.4755 - loss: 8.4541
Epoch 4/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 896us/step - accuracy: 0.4960 - loss: 8.1232
Epoch 5/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 285us/step - accuracy: 0.5165 - loss: 7.7938
Epoch 6/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 346us/step - accuracy: 0.5116 - loss: 7.8725
Epoch 7/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 276us/step - accuracy: 0.4722 - loss: 8.5068
Epoch 8/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 266us/step - accuracy: 0.4832 - loss: 8.3305
Epoch 9/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 314us/step - accuracy: 0.4842 - loss: 8.3143
Epoch 10/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 776us/step - accuracy: 0.4673 - loss: 8.5862
Epoch 11/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 296us/step - accuracy: 0.4888 - loss: 8.2404
Epoch 12/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 337us/step - accuracy: 0.4778 - loss: 8.4173
Epoch 13/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 305us/step - accurac

In [29]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5340 - loss: 7.5109 
Test Loss: 7.8979
Test Accuracy: 0.5100


In [30]:
# Access the weights
weights, bias = model.get_weights()

# Save the weights to a JSON file
weights_dict = {
    "weights": weights.tolist(),
    "bias": bias.tolist()
}

# print(weights)

In [31]:
with open("perceptron_weights.json", "w") as fw:
    json.dump(weights_dict, fw)

print("Weights and bias saved to perceptron_weights.json")

Weights and bias saved to perceptron_weights.json


In [32]:
# Read the JSON file
with open('perceptron_weights.json', 'r') as fr:
    data = json.load(fr)

# Convert the JSON data to a format suitable for Verilog
with open('weights_values.mem', 'w') as fmem:
    for weight_value in data['weights']:
        weight_valueQ15 = weight_value[0] * 2**15 # Convert to Q15.0 fixed-point format
        weight_valueQ15 = int(weight_valueQ15) # Convert to integer
        # Get the raw binary representation
        binary_representation = bin(weight_valueQ15 & 0xFFFF)[2:].zfill(16)
        fmem.write(f"{binary_representation}\n")
        # print(weight_valueQ15)    
        
    bias_value = data['bias']
    bias_valueQ15 = bias_value[0] * 2**15 # Convert to Q15.0 fixed-point format
    bias_valueQ15 = int(bias_valueQ15) # Convert to integer
    # Get the raw binary representation
    binary_representation = bin(bias_valueQ15 & 0xFFFF)[2:].zfill(16)
    fmem.write(f"{binary_representation}\n") 
    

# Save iput data to a file
XQ15 = X * 2**15
XQ15 = XQ15.astype(np.int16)

with open('input_values.mem', 'w') as fmem:
    for value in XQ15[0]:
        binary_representation = bin(value & 0xFFFF)[2:].zfill(16)
        fmem.write(f"{binary_representation}\n")  # Convert value to float before formatting as binary
# print(XQ15)

In [33]:
# Example single input (make sure it has the correct shape)
single_input = X[0].reshape(1, -1)

# Print the input value
# print("Input value for the single input:", single_input)

# Make a prediction
prediction = model.predict(single_input)

predictionQ15 = prediction[0][0] * 2**15 # Convert to Q15.0 fixed-point format

# Print the prediction and the classified class
print("Prediction for the single input:", prediction)
print("Prediction for the single input in Q15.0 format:", int(predictionQ15))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Prediction for the single input: [[0.]]
Prediction for the single input in Q15.0 format: 0


In [34]:
# Assuming single_input and weights are already defined as numpy arrays
# Slice the first ten elements
single_input_ = single_input[0]
weights_ = weights[0:50]

print(single_input_)
print(weights_)

# Perform the dot product with the first ten elements
product = np.dot(single_input_, weights_)
print("Product:", product)
print(product +bias_value[0])

[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864 0.15599452
 0.05808361 0.86617615 0.60111501 0.70807258 0.02058449 0.96990985
 0.83244264 0.21233911 0.18182497 0.18340451 0.30424224 0.52475643
 0.43194502 0.29122914 0.61185289 0.13949386 0.29214465 0.36636184
 0.45606998 0.78517596 0.19967378 0.51423444 0.59241457 0.04645041
 0.60754485 0.17052412 0.06505159 0.94888554 0.96563203 0.80839735
 0.30461377 0.09767211 0.68423303 0.44015249 0.12203823 0.49517691
 0.03438852 0.9093204  0.25877998 0.66252228 0.31171108 0.52006802
 0.54671028 0.18485446]
[[ 0.19429979]
 [-0.2518552 ]
 [-0.12187497]
 [-0.2370421 ]
 [-0.03528419]
 [-0.20235112]
 [ 0.24477002]
 [-0.0190123 ]
 [-0.02542034]
 [ 0.28567526]
 [-0.29212147]
 [ 0.2036213 ]
 [-0.28607422]
 [-0.3392409 ]
 [-0.13261604]
 [-0.30373657]
 [-0.11059721]
 [-0.09488246]
 [ 0.08676782]
 [ 0.08284745]
 [ 0.30256167]
 [ 0.24816981]
 [-0.17766203]
 [ 0.08627471]
 [-0.16686462]
 [ 0.16028878]
 [ 0.23913792]
 [-0.08342183]
 [-0.0427205 ]
 [-0.

In [35]:
# Define the file path
file_path = 'ringer_params.txt'

# Initialize an empty list to store the vector
vector = []

# Open the file and read the lines
with open(file_path, 'r') as file:
    lines = file.readlines()
    # Convert each line to a float and store it in the vector
    vector = [float(line.strip()) for line in lines]

# Print the vector
vector_np = np.array(vector)
sliced_vector = vector_np[::5]
# print(vector_np)
print(sliced_vector)


[ 1.6764456   0.37882406 -0.42926419  0.12437478  2.01618648  1.64988542
 -0.0479631  -1.85738361 -3.67750788 -3.95939612 -3.84060168 -3.79197216
 -3.9009552  -3.47019935 -3.08765268 -3.5244019  -3.27535367 -2.7233429
 -2.89699841 -2.39865112 -2.51283145 -2.41233253 -2.20272136 -1.82715321
 -2.17739701 -0.971668   -0.94159806 -1.02418423 -1.58100104 -1.76152635
 -1.22292805 -2.04296112 -1.30146384 -1.36210108 -1.96421218 -2.1748631
  0.80706775  0.42682707 -0.84317291 -2.10344076 -4.63270473 -3.51675367
 -2.77054715 -2.38905072 -2.45292115 -2.2334826  -1.6267283  -1.76602697
 -0.72634155 -0.50703639  0.09873275  2.87298298  0.24491306]


In [36]:
# Define the file path
file_path_ring = 'rings_data_0.txt'

# Initialize an empty list to store the vector
vector_ring = []

# Open the file and read the lines
with open(file_path_ring, 'r') as file:
    lines = file.readlines()
    # Convert each line to a float and store it in the vector
    vector_ring = [float(line.strip()) for line in lines]

# Print the vector_ring
vector_ring_np = np.array(vector_ring)
# print(vector_ring_np)

# Delete the last three elements
sliced_vector = sliced_vector[:-3]

In [40]:
# Perform element-wise multiplication
result_vector = vector_ring_np * sliced_vector

# Accumulate (sum) all elements
accumulated_result = np.sum(result_vector)

# Print the accumulated result
print("Accumulated result:", accumulated_result)

accumulated_result = accumulated_result + 0.09873274714

print("Accumulated result:", accumulated_result)


Accumulated result: -0.9235487256101009
Accumulated result: -0.8248159784701009


In [38]:
from numpy import trunc
vectorQ15 = vector_np * 2**15
# vectorQ15trunc = trunc(vectorQ15)
vectorQ15trunc = vectorQ15.astype(np.int32)

# print(vectorQ15trunc)


In [39]:
# Convert to two's complement and print in 19-bit binary format
vector_q15_19bit = []
with open('../mem/param.mem', 'w') as mem_file:
    for value in vectorQ15trunc:
        if value < 0:
            # Convert negative value to two's complement
            value = (1 << 19) + value
        # Write the 19-bit binary value to the .mem file
        mem_file.write(format(value, '019b') + '\n')
